<a href="https://colab.research.google.com/github/jeevana-049/MABSA/blob/main/MABSA_step1_caption_ocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pillow requests

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests

# Load the model that describes images (called BLIP)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Download a sample photo from the internet
url = "https://images.unsplash.com/photo-1516035069371-29a1b244cc32?w=600"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# Ask the model to describe it
inputs = processor(image, return_tensors="pt")
output = model.generate(**inputs)
caption = processor.decode(output[0], skip_special_tokens=True)

print("Caption:", caption)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1954: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Caption: a camera and some lenses on a table


In [ ]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 31.7 MB/s eta 0:00:00


In [ ]:
import easyocr
from PIL import Image, ImageDraw, ImageFont

# Make a white picture and write some words on it
img = Image.new("RGB", (600, 200), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.load_default(size=40)
draw.text((30, 70), "BATTERY LOW 10%", fill="black", font=font)
img.save("sample.png")

# Load the text reader (English)
reader = easyocr.Reader(['en'])

# Read the text
results = reader.readtext("sample.png")

for box, text, confidence in results:
    print("Text found:", text, "| Confidence:", round(confidence, 2))

Text found: BATTERY LOW 10% | Confidence: 0.9


In [ ]:
from transformers import pipeline
from PIL import Image
import re

# Sentiment model (gives positive / neutral / negative)
sentiment = pipeline("sentiment-analysis",
                     model="cardiffnlp/twitter-roberta-base-sentiment-latest")

def get_caption(image_path):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(img, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=30)
    return processor.decode(out[0], skip_special_tokens=True)

def get_ocr_text(image_path):
    results = reader.readtext(image_path)
    return " ".join(text for _, text, _ in results)

def filter_by_aspect(text, aspect):
    # Split into small parts and keep only parts that mention the aspect
    parts = re.split(r"[.,;!?]| but | and ", text)
    kept = [p.strip() for p in parts if aspect.lower() in p.lower()]
    return ". ".join(kept) if kept else text

def analyze(text=None, image_path=None, aspect=None):
    pieces = []

    if text:
        pieces.append(text)
    if image_path:
        caption = get_caption(image_path)
        ocr = get_ocr_text(image_path)
        print("Caption:", caption)
        print("OCR text:", ocr if ocr else "(none)")
        pieces.append(caption)
        if ocr:
            pieces.append(ocr)

    if not pieces:
        return "Please give a text, an image, or both."

    mode = "text + image" if (text and image_path) else ("text only" if text else "image only")
    combined = ". ".join(pieces)

    if aspect:
        combined = filter_by_aspect(combined, aspect)

    result = sentiment(combined)[0]
    print(f"Mode: {mode}")
    print(f"Aspect: {aspect if aspect else 'overall'}")
    print(f"Used text: {combined}")
    print(f"Sentiment: {result['label']} ({round(result['score'], 2)})")

# 1) Text only
analyze(text="Great camera, but the delivery was slow.", aspect="camera")
print("-----")
analyze(text="Great camera, but the delivery was slow.", aspect="delivery")
print("-----")

# 2) Image only (uses the sample.png you made in the OCR step)
analyze(image_path="sample.png")
print("-----")

# 3) Both
analyze(text="The battery is terrible.", image_path="sample.png", aspect="battery")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Mode: text only
Aspect: camera
Used text: Great camera
Sentiment: positive (0.92)
-----
Mode: text only
Aspect: delivery
Used text: the delivery was slow
Sentiment: negative (0.85)
-----
Caption: battery low 10 %
OCR text: BATTERY LOW 10%
Mode: image only
Aspect: overall
Used text: battery low 10 %. BATTERY LOW 10%
Sentiment: neutral (0.62)
-----
Caption: battery low 10 %
OCR text: BATTERY LOW 10%
Mode: text + image
Aspect: battery
Used text: The battery is terrible. battery low 10 %. BATTERY LOW 10%
Sentiment: negative (0.95)


In [ ]:
from huggingface_hub import snapshot_download
import os

path = snapshot_download(repo_id="xwycyj/MVSA-Multiple", repo_type="dataset")
print("Saved at:", path)

for root, dirs, files in os.walk(path):
    print(root, "| folders:", len(dirs), "| files:", len(files))
    print(files[:10])
    break

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Saved at: /root/.cache/huggingface/hub/datasets--xwycyj--MVSA-Multiple/snapshots/eb8121066efaa66a5ffb3130f1a119093a780a27
/root/.cache/huggingface/hub/datasets--xwycyj--MVSA-Multiple/snapshots/eb8121066efaa66a5ffb3130f1a119093a780a27 | folders: 0 | files: 2
['test.json', '.gitattributes']


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

path = "/content/drive/MyDrive/MABSA_data/twitter2017/train.tsv"
df = pd.read_csv(path, sep="\t")
df.columns = ["index", "label", "image_id", "text", "aspect"]

label_names = {0: "negative", 1: "neutral", 2: "positive"}
df["sentiment"] = df["label"].map(label_names)
df["text"] = df.apply(lambda r: r["text"].replace("$T$", r["aspect"]), axis=1)

print(df.shape)
print(df["sentiment"].value_counts())
df.head()

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MABSA_data/twitter2017/train.tsv'

In [ ]:
import os
base = "/content/drive/MyDrive"
print("Top level:", os.listdir(base))

for root, dirs, files in os.walk(base):
    if "train.tsv" in files:
        print("FOUND:", os.path.join(root, "train.tsv"))

Top level: ['23AG1A05D5.jpg', 'CLASS Awards 2023 Invitation.pdf', 'Classroom', '23AG1A05E8 (1).jpg', 'IMG-20240220-WA0005.jpg', 'IMG20240326125412.jpg', 'SDC_23AG1A05E8', 'IMG-20250228-WA0001 (1).jpg', 'IMG-20250228-WA0001.jpg', '23AG1A05E8.jpeg', 'er-3c.drawio', 'usecase.drawio', 'class (1).drawio', 'class.drawio', 'broken_weave.drawio', 'bpooo.drawio', '23AG1A05E8.jpg', 'google intern.jpg', 'IMG_20250720_131025.jpg', 'Microsoft  (1)', 'Microsoft ', 'IMG-20250720-WA0001.jpg', 'Colab Notebooks', 'aws.pdf', 'Imp', '063237093784.jpeg', 'Cloud Virtual Internship (1).pdf', 'Cloud Virtual Internship.pdf', '23AG1A05E8 - Jeevana G - CSE.pdf', 'Jeevana_Resumeeee.pdf', 'CRT.jpeg', 'pro_Aceeee.pdf', 'CAW_AI.pdf', 'Jeevana G - Cisco.pdf', 'Gopu Jeevana.pdf', 'MultiModalAI.pdf', 'Algo.zip', 'Screenshot_2026-07-04-14-00-54-23_40deb401b9ffe8e1df2f1cc5ba480b12.jpg', 'Completion Certificate _ SkillsBuild.pdf', 'Screenshot_2026-07-04-14-07-06-65_439a3fec0400f8974d35eed09a31f914.jpg', 'Screenshot_2026-0

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/MABSA_data/train.tsv"
df = pd.read_csv(path, sep="\t")
df.columns = ["index", "label", "image_id", "text", "aspect"]

label_names = {0: "negative", 1: "neutral", 2: "positive"}
df["sentiment"] = df["label"].map(label_names)
df["text"] = df.apply(lambda r: r["text"].replace("$T$", r["aspect"]), axis=1)

print(df.shape)
print(df["sentiment"].value_counts())
df.head()

(3562, 6)
sentiment
neutral     1638
positive    1508
negative     416
Name: count, dtype: int64


,index,label,image_id,text,aspect,sentiment
0,1,2,17_06_10389.jpg,How Jake Paul is changing the influencer game :,Jake Paul,positive
1,2,1,16_05_20_317.jpg,Here are the boys and girls rosters for the in...,Oklahoma,neutral
2,3,0,O_4091.jpg,Chris Brown and his crew were kicked off a pla...,Chris Brown,negative
3,4,1,16_05_03_260.jpg,Stephen Curry wife rejected him first time he ...,Stephen Curry,neutral
4,5,1,16_05_03_260.jpg,Stephen Curry wife rejected him first time he ...,nba,neutral


In [ ]:
from transformers import pipeline

# Reload the sentiment model
sentiment = pipeline("sentiment-analysis",
                     model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Test on 300 random rows
sample = df.sample(300, random_state=42).reset_index(drop=True)

preds = []
for _, row in sample.iterrows():
    out = sentiment(row["text"][:500])[0]
    preds.append(out["label"].lower())

sample["predicted"] = preds
accuracy = (sample["predicted"] == sample["sentiment"]).mean()

print("Accuracy on 300 samples:", round(accuracy * 100, 1), "%")
print(sample[["text", "aspect", "sentiment", "predicted"]].head(10))

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Accuracy on 300 samples: 54.0 %
                                                text           aspect  \
0  Donald Trump ' s New Media Target : Jeff Bezos...     Donald Trump   
1  WATCH : Christina Aguilera ' s impressions of ...            Adele   
2  Paul Pierce has high confidence in Kevin Duran...     Kevin Durant   
3  You are the # summer spirit , you are # Canada...           Canada   
4  Mlb & gt ; The Cincinnati Reds ' bullpen could...  Cincinnati Reds   
5  Donald Trump is the GOP ' s nominee — it ' s t...     Donald Trump   
6  It ' s a great day for Huskies Baseball . Home...  Parkers Prairie   
7  Donald Trump rejects entire intelligence commu...           Russia   
8  RETWEET if you think Lady Gaga should be given...        Lady Gaga   
9  When Shah Rukh Khan turned on the SWAG at Prei...     Preity Zinta   

  sentiment predicted  
0  positive   neutral  
1  positive   neutral  
2  positive   neutral  
3  positive  positive  
4  negative  negative  
5  positive   neutra

In [ ]:
import torch
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

# Split: 80% to train, 20% to check
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42,
                                    stratify=df["label"])

name = "roberta-base"
tok = AutoTokenizer.from_pretrained(name)

class DS(torch.utils.data.Dataset):
    def __init__(self, d):
        # Giving the tweet AND the aspect together makes it aspect-aware
        self.enc = tok(list(d["text"]), list(d["aspect"]), truncation=True,
                       padding="max_length", max_length=96)
        self.labels = list(d["label"])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": float((preds == p.label_ids).mean())}

args = TrainingArguments(output_dir="out", num_train_epochs=3,
                         per_device_train_batch_size=16,
                         per_device_eval_batch_size=32,
                         learning_rate=2e-5, eval_strategy="epoch",
                         save_strategy="no", report_to="none", fp16=True)

trainer = Trainer(model=model, args=args,
                  train_dataset=DS(train_df), eval_dataset=DS(val_df),
                  compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.733961,0.639551
2,No log,0.634006,0.705470
3,0.703150,0.613904,0.723703


Training Loss,Validation Loss,Epoch,Accuracy
0.703150,0.613904,3,0.723703


{'eval_loss': 0.6139037013053894, 'eval_accuracy': 0.7237026647966339}


## Results

| Model | Input | Accuracy |
|---|---|---|
| Pretrained sentiment model (no aspect) | Text | 54.0% |
| Fine-tuned transformer (aspect-aware) | Text + aspect | 72.4% |
| MABSA (with image-to-text) | Text + aspect + image | To be added |

*Dataset: Twitter-2017. Results are on a held-out 20% validation split and will be re-measured on the official test set.*

In [ ]:
import os
folder = "/content/drive/MyDrive/MABSA_data"
for f in os.listdir(folder):
    print(f)

train.tsv
dev.tsv
test.tsv


In [ ]:
import pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, pipeline)
from sklearn.metrics import f1_score, classification_report

folder = "/content/drive/MyDrive/MABSA_data"

# ---------- 1. Load data ----------
def load(name):
    d = pd.read_csv(f"{folder}/{name}.tsv", sep="\t", quoting=3,
                    header=0, names=["index", "label", "image_id", "text", "aspect"])
    d["label"] = d["label"].astype(int)
    d["text"] = d.apply(lambda r: r["text"].replace("$T$", str(r["aspect"])), axis=1)
    return d

train_df, dev_df, test_df = load("train"), load("dev"), load("test")
print("Sizes:", len(train_df), len(dev_df), len(test_df))

# ---------- 2. Baseline: pretrained model, no aspect, official test set ----------
base = pipeline("sentiment-analysis",
                model="cardiffnlp/twitter-roberta-base-sentiment-latest")
name_to_id = {"negative": 0, "neutral": 1, "positive": 2}
base_out = base(list(test_df["text"]), batch_size=32, truncation=True, max_length=128)
base_preds = [name_to_id[o["label"].lower()] for o in base_out]
base_acc = (pd.Series(base_preds) == test_df["label"]).mean()
base_f1 = f1_score(test_df["label"], base_preds, average="macro")
print("Baseline done.")

# ---------- 3. Aspect-aware fine-tuning ----------
tok = AutoTokenizer.from_pretrained("roberta-base")

class DS(torch.utils.data.Dataset):
    def __init__(self, d):
        self.enc = tok(list(d["text"]), list(d["aspect"].astype(str)),
                       truncation=True, padding="max_length", max_length=96)
        self.labels = list(d["label"])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=3)

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": float((preds == p.label_ids).mean()),
            "macro_f1": float(f1_score(p.label_ids, preds, average="macro"))}

args = TrainingArguments(output_dir="out", num_train_epochs=3,
                         per_device_train_batch_size=16,
                         per_device_eval_batch_size=32, learning_rate=2e-5,
                         eval_strategy="epoch", save_strategy="no",
                         report_to="none", fp16=True)

trainer = Trainer(model=model, args=args, train_dataset=DS(train_df),
                  eval_dataset=DS(dev_df), compute_metrics=metrics)
trainer.train()

# ---------- 4. Official test set ----------
out = trainer.predict(DS(test_df))
preds = out.predictions.argmax(-1)
acc = (preds == out.label_ids).mean()
f1 = f1_score(out.label_ids, preds, average="macro")

print("\n========== RESULTS ON OFFICIAL TEST SET ==========")
print(f"Pretrained, no aspect  | Accuracy: {base_acc*100:.1f}% | Macro-F1: {base_f1*100:.1f}%")
print(f"Fine-tuned, with aspect| Accuracy: {acc*100:.1f}% | Macro-F1: {f1*100:.1f}%")
print()
print(classification_report(out.label_ids, preds,
      target_names=["negative", "neutral", "positive"]))

# ---------- 5. Save the trained model to Drive ----------
trainer.save_model(f"{folder}/aspect_model")
tok.save_pretrained(f"{folder}/aspect_model")
print("Model saved to Drive: MABSA_data/aspect_model")

Sizes: 3562 1176 1234


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.706563,0.665816,0.647148
2,No log,0.697330,0.705782,0.693054
3,0.687781,0.714848,0.721088,0.710605



========== RESULTS ON OFFICIAL TEST SET ==========
Pretrained, no aspect  | Accuracy: 56.0% | Macro-F1: 52.8%
Fine-tuned, with aspect| Accuracy: 71.6% | Macro-F1: 70.4%

              precision    recall  f1-score   support

    negative       0.67      0.66      0.67       168
     neutral       0.76      0.67      0.71       573
    positive       0.69      0.79      0.73       493

    accuracy                           0.72      1234
   macro avg       0.71      0.71      0.70      1234
weighted avg       0.72      0.72      0.71      1234



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Drive: MABSA_data/aspect_model


In [ ]:
import pandas as pd
folder = "/content/drive/MyDrive/MABSA_data"

for name in ["train", "dev", "test"]:
    d = pd.read_csv(f"{folder}/{name}.tsv", sep="\t", quoting=3)
    print(name, "| shape:", d.shape)
    print("columns:", list(d.columns))
    print(d.head(3).to_string())
    print("-----")

train | shape: (3562, 5)
columns: ['index', '#1 Label', '#2 ImageID', '#3 String', '#3 String.1']
   index  #1 Label        #2 ImageID                                                                                                      #3 String  #3 String.1
0      1         2   17_06_10389.jpg                                                                      How $T$ is changing the influencer game :    Jake Paul
1      2         1  16_05_20_317.jpg  Here are the boys and girls rosters for the inaugural Native All - State Basketball game in $T$ # NativePreps     Oklahoma
2      3         0        O_4091.jpg                                         $T$ and his crew were kicked off a plane after allegedly hot boxing it  Chris Brown
-----
dev | shape: (1176, 5)
columns: ['index', '#1 Label', '#2 ImageID', '#3 String', '#3 String.1']
   index  #1 Label        #2 ImageID                                                                       #3 String                  #3 String.1
0      1  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip -q install gdown
!gdown --fuzzy "https://drive.google.com/file/d/1PpvvncnQkgDNeBMKVgG2zFYuRhbL873g/view" -O /content/IJCAI2019_data.zip
!ls -lh /content/IJCAI2019_data.zip

Downloading...
From (original): https://drive.google.com/uc?id=1PpvvncnQkgDNeBMKVgG2zFYuRhbL873g
From (redirected): https://drive.google.com/uc?id=1PpvvncnQkgDNeBMKVgG2zFYuRhbL873g&confirm=t&uuid=abee7779-2042-495a-9934-ea1f962bc841
To: /content/IJCAI2019_data.zip
100% 1.03G/1.03G [00:17<00:00, 58.9MB/s]
-rw-r--r-- 1 root root 984M May 23  2019 /content/IJCAI2019_data.zip


In [2]:
!unzip -l /content/IJCAI2019_data.zip | head -n 25

Archive:  /content/IJCAI2019_data.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2019-05-23 05:47   IJCAI2019_data/
        0  2019-01-18 07:09   IJCAI2019_data/twitter2017/
   122833  2018-07-07 01:59   IJCAI2019_data/twitter2017/dev.txt
   367722  2018-07-07 01:59   IJCAI2019_data/twitter2017/train.txt
   129830  2018-07-07 01:59   IJCAI2019_data/twitter2017/test.txt
   383719  2019-01-30 03:10   IJCAI2019_data/twitter2017/train.tsv
   127403  2019-01-30 03:10   IJCAI2019_data/twitter2017/dev.tsv
   134647  2019-01-30 03:10   IJCAI2019_data/twitter2017/test.tsv
        0  2019-05-23 05:55   IJCAI2019_data/twitter2015/
       83  2019-05-23 05:55   IJCAI2019_data/twitter2015/.~lock.train.tsv#
   115293  2019-02-07 13:43   IJCAI2019_data/twitter2015/dev.txt
   107393  2019-02-07 13:43   IJCAI2019_data/twitter2015/test.txt
   324843  2019-02-07 13:43   IJCAI2019_data/twitter2015/train.txt
   338941  2019-02-07 13:44   IJCAI2019_data/twitter2015/trai

In [3]:
!unzip -q /content/IJCAI2019_data.zip "IJCAI2019_data/twitter2017_images/*" -d /content/

import os, pandas as pd
img_dir = "/content/IJCAI2019_data/twitter2017_images"
print("Images unzipped:", len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0)

folder = "/content/drive/MyDrive/MABSA_data"
for n in ["train", "dev", "test"]:
    d = pd.read_csv(f"{folder}/{n}.tsv", sep="\t", quoting=3, header=0,
                    names=["index", "label", "image_id", "text", "aspect"])
    ids = set(d["image_id"])
    have = sum(os.path.exists(f"{img_dir}/{i}") for i in ids)
    print(n, "| unique images needed:", len(ids), "| found:", have)

Images unzipped: 4819


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MABSA_data/train.tsv'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os, pandas as pd
img_dir = "/content/IJCAI2019_data/twitter2017_images"
folder = "/content/drive/MyDrive/MABSA_data"

for n in ["train", "dev", "test"]:
    d = pd.read_csv(f"{folder}/{n}.tsv", sep="\t", quoting=3, header=0,
                    names=["index", "label", "image_id", "text", "aspect"])
    ids = set(d["image_id"])
    have = sum(os.path.exists(f"{img_dir}/{i}") for i in ids)
    print(n, "| unique images needed:", len(ids), "| found:", have)

train | unique images needed: 1745 | found: 1745
dev | unique images needed: 577 | found: 577
test | unique images needed: 587 | found: 587


In [6]:
!pip -q install easyocr
import os, pandas as pd, torch, easyocr
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

folder = "/content/drive/MyDrive/MABSA_data"
img_dir = "/content/IJCAI2019_data/twitter2017_images"
out_csv = f"{folder}/image_text.csv"
dev = "cuda"

# All unique images used by train, dev and test
ids = set()
for n in ["train", "dev", "test"]:
    d = pd.read_csv(f"{folder}/{n}.tsv", sep="\t", quoting=3, header=0,
                    names=["index", "label", "image_id", "text", "aspect"])
    ids |= set(d["image_id"])
ids = sorted(ids)

# Resume if a saved file already exists
done = {}
if os.path.exists(out_csv):
    old = pd.read_csv(out_csv, keep_default_na=False)
    done = {r.image_id: (r.caption, r.ocr) for r in old.itertuples()}
todo = [i for i in ids if i not in done]
print("Total:", len(ids), "| already done:", len(done), "| to do:", len(todo))

proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base").to(dev).half()
reader = easyocr.Reader(["en"], gpu=True)

def open_img(name):
    try:
        return Image.open(f"{img_dir}/{name}").convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), "white")

def save():
    pd.DataFrame([(k, v[0], v[1]) for k, v in done.items()],
                 columns=["image_id", "caption", "ocr"]).to_csv(out_csv, index=False)

B = 16
for s in range(0, len(todo), B):
    batch = todo[s:s + B]
    imgs = [open_img(i) for i in batch]
    inp = proc(images=imgs, return_tensors="pt").to(dev, torch.float16)
    with torch.no_grad():
        out = blip.generate(**inp, max_new_tokens=30)
    caps = proc.batch_decode(out, skip_special_tokens=True)
    for i, c in zip(batch, caps):
        try:
            o = " ".join(reader.readtext(f"{img_dir}/{i}", detail=0))
        except Exception:
            o = ""
        done[i] = (c, o)
    if (s // B) % 10 == 0:
        save()
        print(f"{len(done)}/{len(ids)} done")
save()
print("FINISHED. Saved to", out_csv)
print(pd.read_csv(out_csv, keep_default_na=False).sample(5, random_state=1).to_string())

Total: 2908 | already done: 0 | to do: 2908


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete16/2908 done
176/2908 done
336/2908 done
496/2908 done
656/2908 done
816/2908 done
976/2908 done
1136/2908 done
1296/2908 done
1456/2908 done
1616/2908 done
1776/2908 done
1936/2908 done
2096/2908 done
2256/2908 done
2416/2908 done
2576/2908 done
2736/2908 done
2896/2908 done
FINISHED. Saved to /content/drive/MyDrive/MABSA_data/image_text.csv
              image_id                                                           caption                                                                                                                                                                                                ocr
1265   17_06_14768.jpg  a plate of ta ta ta ta ta ta ta ta ta ta ta ta ta ta ta ta ta ta                                                                                                                                                                                                   
2765     

In [7]:
import pandas as pd, torch

df_it = pd.read_csv(out_csv, keep_default_na=False)

def bad(c):
    w = c.split()
    return len(w) < 3 or len(set(w)) < 0.6 * len(w)

mask = df_it["caption"].apply(bad)
print("Bad captions:", mask.sum(), "of", len(df_it))
print("Images with OCR text:", (df_it["ocr"].str.len() > 0).sum())

blip = blip.float()          # switch to full precision
fix = df_it[mask]["image_id"].tolist()

for s in range(0, len(fix), 8):
    batch = fix[s:s + 8]
    imgs = [open_img(i) for i in batch]
    inp = proc(images=imgs, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = blip.generate(**inp, max_new_tokens=30,
                            num_beams=3, no_repeat_ngram_size=2)
    caps = proc.batch_decode(out, skip_special_tokens=True)
    for i, c in zip(batch, caps):
        df_it.loc[df_it["image_id"] == i, "caption"] = c

df_it.to_csv(out_csv, index=False)
print("Bad after fix:", df_it["caption"].apply(bad).sum())
print(df_it.sample(5, random_state=3)[["image_id", "caption", "ocr"]].to_string())

Bad captions: 180 of 2908
Images with OCR text: 1800
Bad after fix: 2
              image_id                                            caption                                                                                                                                                                                                                                                   ocr
2380        O_3274.jpg             a basketball player jumping in the air                                                                                                                                                                                                                                                      
2641        O_4358.jpg      barcelona ' s neer is the latest player to be                                                                                                                                                                                                                         

In [8]:
import re, gc, torch, pandas as pd
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import f1_score

folder = "/content/drive/MyDrive/MABSA_data"

# ---------- Load text data + image text ----------
def load(name):
    d = pd.read_csv(f"{folder}/{name}.tsv", sep="\t", quoting=3, header=0,
                    names=["index", "label", "image_id", "text", "aspect"])
    d["label"] = d["label"].astype(int)
    d["text"] = d.apply(lambda r: r["text"].replace("$T$", str(r["aspect"])), axis=1)
    d["aspect"] = d["aspect"].astype(str)
    return d

train_df, dev_df, test_df = load("train"), load("dev"), load("test")
it = pd.read_csv(f"{folder}/image_text.csv", keep_default_na=False).set_index("image_id")
for d in (train_df, dev_df, test_df):
    d["cap"] = d["image_id"].map(it["caption"]).fillna("")
    d["ocr"] = d["image_id"].map(it["ocr"]).fillna("")

# ---------- Two ways to build the visual text ----------
STOP = set("a an the is of in on at to and with for this that it his her their its are was be by as from or up".split())
def words(s):
    return {w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP and len(w) > 2}

def visual_raw(r):
    return (r["cap"] + " . " + " ".join(r["ocr"].split()[:40])).strip()

def visual_filtered(r):
    key = words(r["aspect"]) | words(r["text"])
    w = r["ocr"].split()
    chunks = [" ".join(w[i:i + 8]) for i in range(0, min(len(w), 64), 8)]
    kept = [c for c in chunks if words(c) & key]
    return (r["cap"] + " . " + " ".join(kept)).strip()

tok = AutoTokenizer.from_pretrained("roberta-base")

class DS(torch.utils.data.Dataset):
    def __init__(self, d, vis_fn):
        second = [f"{r['aspect']} | {vis_fn(r)}" for _, r in d.iterrows()]
        self.enc = tok(list(d["text"]), second, truncation=True,
                       padding="max_length", max_length=192)
        self.labels = list(d["label"])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

def metrics(p):
    pr = p.predictions.argmax(-1)
    return {"accuracy": float((pr == p.label_ids).mean()),
            "macro_f1": float(f1_score(p.label_ids, pr, average="macro"))}

results = {}
for name, fn in [("A: text+aspect+caption+OCR (no filter)", visual_raw),
                 ("B: text+aspect+caption+filtered OCR", visual_filtered)]:
    print("\n=====", name, "=====")
    model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=3)
    args = TrainingArguments(output_dir="out", num_train_epochs=3,
                             per_device_train_batch_size=16,
                             per_device_eval_batch_size=32, learning_rate=2e-5,
                             eval_strategy="epoch", save_strategy="no",
                             report_to="none", fp16=True, seed=42)
    tr = Trainer(model=model, args=args, train_dataset=DS(train_df, fn),
                 eval_dataset=DS(dev_df, fn), compute_metrics=metrics)
    tr.train()
    out = tr.predict(DS(test_df, fn))
    pr = out.predictions.argmax(-1)
    results[name] = ((pr == out.label_ids).mean() * 100,
                     f1_score(out.label_ids, pr, average="macro") * 100)
    del model, tr
    gc.collect(); torch.cuda.empty_cache()

print("\n========== OFFICIAL TEST SET ==========")
print("Text + aspect only (earlier run): Accuracy 71.6% | Macro-F1 70.4%")
for k, (a, f) in results.items():
    print(f"{k}: Accuracy {a:.1f}% | Macro-F1 {f:.1f}%")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...


===== B: text+aspect+caption+filtered OCR =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.814451,0.587585,0.550568
2,No log,0.694919,0.694728,0.675790
3,0.741080,0.731018,0.704932,0.686494



========== OFFICIAL TEST SET ==========
Text + aspect only (earlier run): Accuracy 71.6% | Macro-F1 70.4%
A: text+aspect+caption+OCR (no filter): Accuracy 71.1% | Macro-F1 69.6%
B: text+aspect+caption+filtered OCR: Accuracy 72.5% | Macro-F1 70.9%
